# 第 07 章　XGBoost 与 SHAP

> **本章问题**：XGBoost 比随机森林多做了什么？SHAP 又能告诉我们什么？

随机森林把很多棵树的预测平均起来。XGBoost 则是一棵接一棵地补前面模型留下的误差。它可能更准，也可能更容易过拟合，所以调参还是只看训练区的空间折，测试区最后才打开。

模型定下来以后，我们再用 SHAP 看看预测如何分摊到各个输入变量上。先把边界讲清楚：SHAP 解释模型在做什么，不等于解释真实地理过程，更不能单独证明因果关系。


## 7.1　从随机森林走到梯度提升

可以把 XGBoost 想成一组接力树：新树重点补前面模型还没解释好的部分。`learning_rate` 控制每棵树迈多大一步，`n_estimators` 是树的数量，`max_depth` 控制每棵树能切得多细。

这几个旋钮需要一起看。学习率小、树多，通常训练更慢；树很深、学习率又大，则更容易把训练区的局部特点记下来。本章只试 18 组参数，重点是学会用空间折调参。


In [ ]:

import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.text as mtext
from sklearn.model_selection import GridSearchCV, PredefinedSplit
import shap

PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT / "src"))
import geoml_data as gd
import geoml_models as gm
import geoml_style as gs

gs.setup()
np.random.seed(gm.SEED)
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)

print("Python:", sys.executable)
print("CONDA_PREFIX:", os.environ.get("CONDA_PREFIX"))
print("XGBoost / SHAP:", __import__("xgboost").__version__, "/", shap.__version__)


## 7.2　固定空间折，并给变量换上中文名

为了让 SHAP 图直接显示中文，本章把特征列改名为“高程、NDVI、气温、降水、土壤水分”。这里只改列名，不改数据和像元顺序，训练测试划分也和前几章相同。

测试区的 1360 个像元继续锁好。我们不会用它挑参数，只在模型定下来以后评估一次。


In [ ]:

df = gd.load_splits()
train = df[df["split"] == "train"].copy()
test = df[df["split"] == "test"].copy()

cn = gd.FEATURE_LABELS
train_cn = train.rename(columns=cn)
test_cn = test.rename(columns=cn)
features = [cn[k] for k in gm.FEATURES]

X_train, y_train = train_cn[features], train_cn["lst"]
X_test, y_test = test_cn[features], test_cn["lst"]

print("训练区 / 测试区：", len(train), "/", len(test))
print("特征：", features)
print("训练区空间折像元数：", train["fold"].value_counts().sort_index().to_dict())
display(train_cn[["row", "col", "lst", *features]].head())


## 7.3　只在训练区的空间折里调参

`PredefinedSplit` 会按照已有的 `fold` 编号轮流留出验证块，`GridSearchCV` 则负责跑完参数组合。评分函数用负 RMSE，这是 scikit-learn 的记分约定，表格里已经转回正的 RMSE，越低越好。

所以这里找到的是“训练区空间折里表现最好的一组”，不是测试区最好的参数。两者要分清。


In [ ]:

param_grid = {
    "max_depth": [2, 3, 4],
    "learning_rate": [0.03, 0.05, 0.1],
    "n_estimators": [200, 400],
}

predefined = PredefinedSplit(test_fold=train["fold"].to_numpy())
search = GridSearchCV(
    estimator=gm.xgb_model(),
    param_grid=param_grid,
    scoring="neg_root_mean_squared_error",
    cv=predefined,
    refit=True,
    n_jobs=1,
    verbose=0,
)
t0 = time.perf_counter()
search.fit(X_train, y_train)
tune_seconds = time.perf_counter() - t0

tuning = pd.DataFrame(search.cv_results_).copy()
tuning["RMSE_折均"] = -tuning["mean_test_score"]
tuning["RMSE_折标准差"] = tuning["std_test_score"]
tuning = tuning.rename(columns={
    "param_max_depth": "max_depth",
    "param_learning_rate": "learning_rate",
    "param_n_estimators": "n_estimators",
    "rank_test_score": "排名",
})
tuning = tuning[["max_depth", "learning_rate", "n_estimators", "RMSE_折均", "RMSE_折标准差", "排名"]]
tuning = tuning.sort_values("排名").reset_index(drop=True)
tuning.to_csv(gd.TABLES / "07_tuning.csv", index=False, encoding="utf-8-sig")

best_params = search.best_params_
print(f"调参完成：{len(tuning)} 个组合 × 5 折，用时 {tune_seconds:.1f}s")
print("最佳参数：", best_params)
print(f"最佳训练区 CV RMSE：{-search.best_score_:.4f}")
print("\n前 10 名：")
display(tuning.head(10).round(4))



从表中要看的不是“第一名”三个字，而是：

- 最优区域是否连着出现。如果相邻的树深/学习率都不差，说明结论比较稳；如果只有孤立一个格最好，可能只是折的偶然。
- 折间标准差有多大。均值只差 0.01、标准差却有 0.2 时，不必迷信排名。
- 最优树深是不是落在网格边缘。如果 `max_depth=4` 最好，应该警惕“再深会不会更好”，但不能再拿测试区去试。


## 7.4　把 XGBoost 和随机森林放到同一把尺子上

随机森林沿用第 06 章的配置，XGBoost 用刚才选出的参数。两者都先在全部训练区重新拟合，再预测同一个独立测试区。

现在只是做一次公平比较，不再根据测试结果回头调参数。


In [ ]:

best_xgb_params = dict(best_params)
xgb_cv = gm.cv_scores(lambda: gm.xgb_model(**best_xgb_params), train_cn, features)
rf_cv = gm.cv_scores(lambda: gm.forest_model(), train_cn, features)

xgb = gm.xgb_model(**best_xgb_params)
t0 = time.perf_counter()
xgb.fit(X_train, y_train)
xgb_pred = xgb.predict(X_test)
xgb_seconds = time.perf_counter() - t0
xgb_test = gm.metrics(y_test, xgb_pred)

rf = gm.forest_model()
t0 = time.perf_counter()
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_seconds = time.perf_counter() - t0
rf_test = gm.metrics(y_test, rf_pred)

fold_scores = pd.concat(
    [xgb_cv.assign(模型="XGBoost"), rf_cv.assign(模型="随机森林")],
    ignore_index=True,
)
fold_scores.to_csv(gd.TABLES / "07_fold_scores.csv", index=False, encoding="utf-8-sig")

metrics = pd.DataFrame([
    {"模型": "XGBoost", "MAE_CV": xgb_cv.MAE.mean(), "RMSE_CV": xgb_cv.RMSE.mean(),
     "R2_CV": xgb_cv.R2.mean(), "MAE_测试": xgb_test["MAE"],
     "RMSE_测试": xgb_test["RMSE"], "R2_测试": xgb_test["R2"], "耗时秒": xgb_seconds,
     "参数": str(best_xgb_params)},
    {"模型": "随机森林", "MAE_CV": rf_cv.MAE.mean(), "RMSE_CV": rf_cv.RMSE.mean(),
     "R2_CV": rf_cv.R2.mean(), "MAE_测试": rf_test["MAE"],
     "RMSE_测试": rf_test["RMSE"], "R2_测试": rf_test["R2"], "耗时秒": rf_seconds,
     "参数": "400 棵，min_samples_leaf=3"},
])
metrics.to_csv(gd.TABLES / "07_metrics.csv", index=False, encoding="utf-8-sig")

pred_out = test[["row", "col", "x", "y", "lst"]].rename(columns={"lst": "obs"})
pred_out["pred_xgb"] = xgb_pred
pred_out["pred_rf"] = rf_pred
pred_out.to_csv(gd.TABLES / "07_test_predictions.csv", index=False, encoding="utf-8-sig")

print(f"XGBoost   CV RMSE={xgb_cv.RMSE.mean():.4f}  测试 RMSE={xgb_test['RMSE']:.4f}  "
      f"R²={xgb_test['R2']:.3f}  用时={xgb_seconds:.2f}s")
print(f"随机森林  CV RMSE={rf_cv.RMSE.mean():.4f}  测试 RMSE={rf_test['RMSE']:.4f}  "
      f"R²={rf_test['R2']:.3f}  用时={rf_seconds:.2f}s")
display(metrics.round(3))


In [ ]:

fold_rmse = {
    "XGBoost": xgb_cv["RMSE"].to_numpy(),
    "随机森林": rf_cv["RMSE"].to_numpy(),
}
test_rmse = {"XGBoost": xgb_test["RMSE"], "随机森林": rf_test["RMSE"]}
gs.model_comparison(
    fold_rmse,
    test_rmse,
    out_name="07_model_compare.png",
    title="XGBoost 与随机森林：相同空间折与测试区",
    ylabel="RMSE（原始像元值）",
)
print("图件：outputs/figures/07_model_compare.png")


## 7.5　SHAP：拆开看一个模型的预测

SHAP 会把预测值相对基线的差异，分摊给各个特征。先看 beeswarm，每个点代表一个像元，横轴是贡献方向和大小，颜色表示特征取值高低。再看 waterfall，我们挑一个留出区像元，逐步拆开它的预测。

请记住，这些图回答的是“这个模型用了什么信息”。它们不证明气温导致 LST 变化。若两个输入变量相关，模型可能用其中一个代替另一个，贡献也会跟着分摊。


In [ ]:

explainer = shap.TreeExplainer(xgb)
shap_train = explainer.shap_values(X_train)
shap_test = explainer.shap_values(X_test)
print("SHAP 矩阵：", shap_train.shape, shap_test.shape)
print("基线 E[f(X)]：", float(np.ravel(explainer.expected_value)[0]))

importance = pd.DataFrame({
    "特征": features,
    "平均绝对SHAP": np.abs(shap_train).mean(axis=0),
}).sort_values("平均绝对SHAP", ascending=False)
importance.to_csv(gd.TABLES / "07_shap_importance.csv", index=False, encoding="utf-8-sig")
display(importance.round(4))


In [ ]:

shap.summary_plot(shap_train, X_train, show=False, max_display=len(features))
fig = plt.gcf()
fig.set_size_inches(7.6, 4.5)
ax = fig.axes[0]
ax.set_title(f"SHAP 全局归因（训练区，n={len(X_train)}）", fontsize=13, pad=12)
ax.set_xlabel("SHAP 值（对预测的影响）", fontsize=10.5)
cbar = fig.axes[-1]
cbar.set_ylabel("特征取值", fontsize=10)
cbar.set_yticks([0, 1])
cbar.set_yticklabels(["低", "高"], fontsize=9)
fig.tight_layout()
gs.save(fig, "07_beeswarm.png")
plt.show()

print("平均绝对 SHAP 最大的变量：", importance.iloc[0]["特征"],
      f"{importance.iloc[0]['平均绝对SHAP']:.3f}")


In [ ]:

# 选一个预测值接近测试区中位数的留出区像元，读它的 SHAP 分解
water_idx = int(np.argmin(np.abs(xgb_pred - np.median(xgb_pred))))
row_i = test.iloc[water_idx]
x_one = X_test.iloc[[water_idx]]
sv_one = explainer.shap_values(x_one)
base_value = float(np.ravel(explainer.expected_value)[0])
explanation = shap.Explanation(
    values=sv_one[0],
    base_values=base_value,
    data=x_one.iloc[0].to_numpy(),
    feature_names=features,
)

shap.plots.waterfall(explanation, max_display=len(features), show=False)
fig = plt.gcf()
fig.set_size_inches(7.4, 4.5)
ax = fig.axes[0]
for txt in fig.findobj(mtext.Text):
    if "E[f(X)]" in txt.get_text() or "11.948" in txt.get_text():
        txt.set_visible(False)
ax.set_xlabel("模型输出（LST 原始像元值）", fontsize=10.5, labelpad=8)
ax.set_title(
    f"留出区像元 row={int(row_i['row'])}, col={int(row_i['col'])} 的 SHAP 分解\n"
    f"观测 {row_i['lst']:.2f}，预测 {xgb_pred[water_idx]:.2f}，训练区基线 {base_value:.2f}",
    fontsize=11.5, pad=10,
)
fig.tight_layout()
gs.save(fig, "07_waterfall.png")
plt.show()

contrib = pd.Series(sv_one[0], index=features).sort_values(key=np.abs, ascending=False)
print(f"像元位置：row={int(row_i['row'])}, col={int(row_i['col'])}")
print(f"观测 LST={row_i['lst']:.3f}，预测 LST={xgb_pred[water_idx]:.3f}，基线={base_value:.3f}")
print("SHAP 贡献（正=推高预测，负=压低预测）：")
display(contrib.to_frame("SHAP").round(3))
print("校验：基线 + 全部 SHAP =", round(base_value + contrib.sum(), 4))



waterfall 的读法：

- 最上方是模型对该像元的最终输出，最下方是训练区基线；
- 红色条把预测推高，蓝色条把预测压低；
- 每条左侧的数值是该像元的特征取值，右侧是 SHAP 贡献；
- 条的长度是“相对基线”的贡献，不是该变量单独作用下的 LST 变化量。

如果一个像元的气温不高、但高程也低，模型可能给出“气温压低、高程推高”的组合；这正是模型在训练分布内做的补偿。换一个像元，分解会变。


## 7.6　相关变量会影响 SHAP 怎么分账

训练区里，高程和气温的相关系数约 0.32，高程和降水约 0.28。相关性不算特别高，但模型仍可能在它们之间选择替代变量。

所以看完 SHAP 排名，建议把结论说完整：“在当前模型、输入变量和训练样本中，气温的平均绝对贡献最大。”这比“气温最重要”严谨得多。下面的相关矩阵就是提醒我们，归因需要结合变量关系一起读。


In [ ]:

corr = train_cn[features].corr()
fig, ax = plt.subplots(figsize=(5.8, 4.8), constrained_layout=True)
im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(features)))
ax.set_xticklabels(features, rotation=30, ha="right")
ax.set_yticks(range(len(features)))
ax.set_yticklabels(features)
for i in range(len(features)):
    for j in range(len(features)):
        ax.text(j, i, f"{corr.values[i, j]:.2f}", ha="center", va="center",
                fontsize=9, color="white" if abs(corr.values[i, j]) > 0.55 else "#222222")
ax.set_title("训练区预测变量相关性（Pearson r）")
cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03)
cb.set_label("相关系数")
gs.save(fig, "07_correlation.png")
plt.show()

gd.write_json({
    "notebook": "07_XGBoost与SHAP.ipynb",
    "kernel": "geoml",
    "python": sys.executable,
    "conda_prefix": os.environ.get("CONDA_PREFIX"),
    "tuning_seconds": float(tune_seconds),
    "best_params": {k: (int(v) if isinstance(v, (np.integer,)) else float(v) if isinstance(v, (np.floating,)) else v)
                    for k, v in best_params.items()},
    "best_cv_rmse": float(-search.best_score_),
    "models": metrics.round(6).to_dict(orient="records"),
    "waterfall_pixel": {
        "row": int(row_i["row"]), "col": int(row_i["col"]),
        "observed": float(row_i["lst"]), "predicted": float(xgb_pred[water_idx]),
        "base_value": float(base_value),
        "top_contributions": {k: float(v) for k, v in contrib.head(5).items()},
    },
    "shap_importance": importance.round(6).to_dict(orient="records"),
    "unverified": [
        "LST/降水/土壤水分单位未在元数据中注明，统一按原始像元值。",
        "LST 2021 与降水/土壤水分 2020 不同期。",
        "SHAP 是模型归因，不是因果效应；相关变量会分摊贡献。",
    ],
}, "07_run_summary.json")



本次运行：XGBoost 的训练区 CV RMSE **1.278**、测试 RMSE **1.432**、测试 R² **0.422**；随机森林分别为 **1.299**、**1.471**、**0.390**。测试 RMSE 低约 0.040（2.7%），方向一致，但优势不大；XGBoost 多出的调参成本是否值得，要结合项目目标判断。

## 7.7 结果怎么读

把三张图放在一起：

1. **模型比较**：XGBoost 在训练区空间 CV 和独立测试区上是否都比随机森林好？如果测试区优势很小，是否值得多出来的调参成本？
2. **beeswarm**：气温的 SHAP 范围最宽，说明它在这个模型里贡献最大；但颜色顺序是否单调？如果高气温既有正贡献也有负贡献，说明存在交互或相关变量的补偿。
3. **waterfall**：一个像元的预测是多项贡献叠加的结果。不要把它读成“某变量单独造成了多少度”。

测试区与训练区的 LST 均值不同（12.83 vs 11.95），这会带来系统偏差。即便 RMSE 看起来可接受，预测图也可能整体偏冷或偏暖；第 08 章会用三联图继续检查。


## 7.8　带着地理问题读结果

- 树模型会把特征空间切成许多小区域，擅长在见过的范围内查找相似组合，但对没见过的组合不擅长外推。
- 高程、气温、降水和土壤水分都有各自的时间与空间尺度。本例里的部分变量年份不同，别把模型贡献讲成同期作用机制。
- 空间折让验证更接近“去一个新区域预测”的任务，因此调参沿用空间折。
- SHAP 帮我们检查模型如何组合输入，真实地理机制还需要独立证据。
